# Семинар №9. Транзакции, MVCC, изоляция и блокировки

## Как устроен ноутбук

1. Каждая тема идёт по схеме: **теория → демо → упражнение**.
2. Ячейки с комментарием `# УПРАЖНЕНИЕ N` — для самостоятельной работы; там стоят `TODO`.
3. Выполняйте ячейки **по порядку**: состояние таблиц в БД важно.
4. Сброс: `Kernel → Restart & Run All`.

## Окружение

- PostgreSQL 16 слушает `localhost:5432` **внутри контейнера**.
- Учётные данные БД: пользователь `postgres`, база `demo`. Если в вашем окружении включён пароль — добавьте `password="..."` в словарь `DB` ниже.
- Jupyter-пароль: `student`.

## План

1. Как устроена транзакционность: XID, CLOG, wraparound, freeze.
2. Компромисс консистентность vs доступность; MVCC.
3. Распределённые транзакции: 2PC и Saga.
4. Уровни изоляции в PostgreSQL.
5. Best practices по транзакциям.
6. Пессимистичные блокировки (`SELECT … FOR UPDATE`) и SQLAlchemy.
7. Оптимистичный контроль версий.
8. Deadlock и его диагностика.


In [ ]:
# Проверка окружения и общая конфигурация
import sys
print("Python:", sys.version.split()[0])

import psycopg, sqlalchemy, pandas, numpy
print("psycopg     :", psycopg.__version__)
print("sqlalchemy  :", sqlalchemy.__version__)
print("pandas      :", pandas.__version__)
print("numpy       :", numpy.__version__)

DB = {
    "host":   "localhost",
    "port":   5432,
    "dbname": "demo",
    "user":   "postgres",
}

PSYCOPG_CONN = (
    f"host={DB['host']} port={DB['port']} dbname={DB['dbname']} "
    f"user={DB['user']}"
)
SQLALCHEMY_URL = (
    f"postgresql+psycopg://{DB['user']}@{DB['host']}:{DB['port']}/{DB['dbname']}"
)

from sqlalchemy import create_engine, text
engine = create_engine(SQLALCHEMY_URL, echo=False)

print("\npsycopg DSN      :", PSYCOPG_CONN)
print("SQLAlchemy URL   :", SQLALCHEMY_URL)

# Быстрая проверка подключения
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print("\nPG version:", cur.fetchone()[0].split(" on ")[0])


## 2.1 Как устроена транзакционность в PostgreSQL

Когда транзакция выполняет первую **изменяющую** операцию (`INSERT/UPDATE/DELETE`) или явно запрашивает XID через `SELECT txid_current()`, PostgreSQL назначает ей уникальный **XID (Transaction ID)**. Пустой `BEGIN; SELECT 1; COMMIT;` **не получает XID вообще** — это важно, потому что 32-битное пространство XID конечно (~4 млрд значений).

- Каждая строка в heap имеет заголовок с `xmin` (XID вставившей транзакции) и `xmax` (XID удалившей/изменившей).
- Статус транзакции (IN_PROGRESS / COMMITTED / ABORTED / SUBCOMMITTED) хранится в **CLOG** — компактном массиве 2-битных состояний (каталог `pg_xact`).
- **`SELECT` без `FOR UPDATE` не блокирует и не блокируется `UPDATE`/`DELETE`** — читатель работает со снапшотом, а не с блокировками строк.

### Проблема wraparound и freeze

XID цикличен. Если старые строки не «заморозить», после переполнения счётчика они станут «из будущего» и пропадут из видимости. Autovacuum заменяет их на `FrozenTransactionId`, который всегда считается старым. Долгие транзакции мешают vacuum → риск wraparound и аварийной остановки.

**Золотое правило: транзакции должны быть короткими.**


In [ ]:
# Демо: когда XID присваивается, а когда нет
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT txid_current_if_assigned()")
        print("XID в начале транзакции (до изменений):", cur.fetchone()[0])
    conn.commit()

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS xid_demo")
        cur.execute("CREATE TABLE xid_demo (id INT PRIMARY KEY, val INT)")
        cur.execute("INSERT INTO xid_demo VALUES (1, 100)")
        cur.execute("SELECT txid_current_if_assigned()")
        print("XID после INSERT:", cur.fetchone()[0])
    conn.commit()

# Проверим xmin/xmax у строки
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT id, val, xmin, xmax FROM xid_demo")
        print("\n(id, val, xmin, xmax):", cur.fetchone())

# Параметры autovacuum, связанные с freeze
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT name, setting FROM pg_settings
            WHERE name IN ('autovacuum', 'autovacuum_freeze_max_age')
        """)
        for name, val in cur.fetchall():
            print(f"{name:30s} = {val}")


### Упражнение 1. Разберитесь с XID и активными снапшотами

**Задание.** Продемонстрируйте, что `txid_current_if_assigned()` возвращает `NULL` до изменений, а после `INSERT` — реальный XID. Дополнительно посмотрите, что видно в `pg_stat_activity` по вашей сессии: `backend_xid` (XID, назначенный текущей транзакции) и `backend_xmin` (нижняя граница снапшота).

**Шаги.**
1. В новой транзакции выполните `SELECT txid_current_if_assigned()` — напечатайте результат.
2. Сделайте `INSERT` в `xid_demo` (используйте id 100).
3. Снова выполните `SELECT txid_current_if_assigned()` — значение должно быть не NULL.
4. Из **второго** соединения (`psycopg.connect(...)`) запросите `pg_stat_activity` по `pid = pg_backend_pid()` первой сессии: посмотрите `backend_xid`, `backend_xmin`.
5. Сделайте `COMMIT` в первой сессии и повторите п.4 — что изменилось?

**Критерий успеха.** До `INSERT` — `None`, после — число; `backend_xid` виден, пока транзакция открыта.


In [3]:
# УПРАЖНЕНИЕ 1

# TODO: подключитесь, проверьте txid_current_if_assigned() до изменений
# TODO: сделайте INSERT в xid_demo
# TODO: снова проверьте XID
# TODO: из второго соединения посмотрите pg_stat_activity (backend_xid, backend_xmin)
# TODO: commit и повторите


## 2.2 Компромисс: консистентность vs доступность/производительность

Строгая изоляция требует координации, координация снижает пропускную способность. PostgreSQL выбрал **MVCC (Multi-Version Concurrency Control)**: читатели не блокируют писателей, создаются новые версии строк.

Плюсы MVCC:
- высокая пропускная способность чтения;
- читатели не видят «грязных» данных;
- нет read-блокировок.

Минусы MVCC:
- нужен периодический **VACUUM** (autovacuum) для очистки мёртвых версий;
- **bloat** таблиц при долгих транзакциях;
- риск wraparound, если vacuum не успевает.

**Ключевая мысль:** долгая транзакция держит старый снапшот и мешает vacuum удалить старые версии. Поэтому её влияние выходит за рамки «моей» таблицы — она бьёт по всей БД.


In [ ]:
# Демо: активная транзакция держит backend_xmin
with psycopg.connect(PSYCOPG_CONN) as conn1:
    with conn1.cursor() as cur1:
        cur1.execute("BEGIN ISOLATION LEVEL REPEATABLE READ")
        cur1.execute("SELECT count(*) FROM xid_demo")  # создаёт snapshot
        cur1.fetchone()
        pid1 = None
        cur1.execute("SELECT pg_backend_pid()")
        pid1 = cur1.fetchone()[0]
        print("Сессия 1 pid =", pid1)

        # Из второго соединения смотрим pg_stat_activity
        with psycopg.connect(PSYCOPG_CONN) as conn2:
            with conn2.cursor() as cur2:
                cur2.execute(
                    "SELECT pid, state, backend_xid, backend_xmin "
                    "FROM pg_stat_activity WHERE pid = %s", (pid1,)
                )
                print("\nПока транзакция 1 открыта:")
                print(cur2.fetchone())
    conn1.rollback()

# Транзакция закрыта — backend_xmin освобождён
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT pid, state, backend_xid, backend_xmin "
            "FROM pg_stat_activity WHERE pid = %s", (pid1,)
        )
        print("\nПосле rollback:")
        print(cur.fetchone())


### Упражнение 2. Долгая транзакция мешает vacuum

**Задание.** Покажите, что `backend_xmin` держится всё время жизни транзакции и освобождается только после её завершения. Это и есть причина, по которой долгие транзакции мешают vacuum.

**Шаги.**
1. Откройте соединение `conn1`, начните `BEGIN ISOLATION LEVEL REPEATABLE READ` и выполните `SELECT` из `xid_demo` — создайте snapshot.
2. Из второго соединения посмотрите `pg_stat_activity` по `pid` первой сессии: `backend_xmin` должен быть не NULL.
3. Сделайте в первой сессии ещё один `SELECT` из `xid_demo` — `backend_xmin` не меняется (snapshot стабилен).
4. Закройте транзакцию `ROLLBACK` и снова проверьте `pg_stat_activity` — `backend_xmin` должен стать NULL.
5. В комментарии к коду напишите 1–2 строки: почему удержание `backend_xmin` блокирует очистку мёртвых версий?

**Критерий успеха.** В консоли видно `backend_xmin: <число>` пока транзакция открыта, и `None` после закрытия.


In [5]:
# УПРАЖНЕНИЕ 2

# TODO: conn1 + BEGIN REPEATABLE READ + SELECT
# TODO: из conn2 посмотреть pg_stat_activity
# TODO: второй SELECT в conn1 — backend_xmin не меняется
# TODO: rollback и снова проверить
# TODO: комментарий: почему это блокирует vacuum


## 2.3 Транзакции в распределённых системах

Когда данные размазаны по нескольким базам (микросервисам) или шардам, классические ACID-транзакции «из коробки» не работают. Два основных подхода:

### 2PC (Two-Phase Commit)

- **Фаза 1 (Prepare):** координатор спрашивает все узлы: «Готовы закоммитить?». Узлы блокируют данные и отвечают «Да/Нет».
- **Фаза 2 (Commit/Rollback):** если все «Да» — `COMMIT`, иначе `ROLLBACK`.
- **Минус:** падение координатора после Prepare оставляет участников в заблокированном состоянии → страдает доступность.
- PostgreSQL поддерживает 2PC через `PREPARE TRANSACTION` / `COMMIT PREPARED` / `ROLLBACK PREPARED` (требуется `max_prepared_transactions > 0`).

### Saga + Eventual Consistency

- Вместо одной распределённой транзакции — цепочка локальных.
- При ошибке шага N запускаются компенсирующие транзакции (отменяющие 1…N-1).
- Данные временно неконсистентны, но система Available. Стандарт для высоконагруженных микросервисов.
- **Оркестрация** — центральный координатор. **Хореография** — сервисы подписываются на события друг друга через брокер (Kafka, RabbitMQ).


In [ ]:
# Демо: PREPARE TRANSACTION (2PC) в PostgreSQL
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS twopc_demo")
        cur.execute("CREATE TABLE twopc_demo (id INT PRIMARY KEY, val INT)")
        cur.execute("INSERT INTO twopc_demo VALUES (1, 100)")
        cur.execute("SHOW max_prepared_transactions")
        max_prep = int(cur.fetchone()[0])
        print("max_prepared_transactions =", max_prep)
        if max_prep == 0:
            print("PREPARE TRANSACTION недоступен (нужно max_prepared_transactions > 0 и перезапуск PG).")
        else:
            # Явно готовим транзакцию
            cur.execute("BEGIN")
            cur.execute("UPDATE twopc_demo SET val = 200 WHERE id = 1")
            cur.execute("PREPARE TRANSACTION 'demo_prepared'")
            print("Prepared.")

# Смотрим список подготовленных транзакций
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT gid, prepared, owner, database FROM pg_prepared_xacts")
        rows = cur.fetchall()
        print("pg_prepared_xacts:", rows)

# Финализируем (в реальной системе это делает координатор)
if rows:
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute("COMMIT PREPARED 'demo_prepared'")
            cur.execute("SELECT val FROM twopc_demo WHERE id = 1")
            print("After COMMIT PREPARED: val =", cur.fetchone()[0])


### Упражнение 3. PREPARE TRANSACTION и откат

**Задание.** Проведите полный цикл 2PC-транзакции: подготовьте, посмотрите в `pg_prepared_xacts`, откатите через `ROLLBACK PREPARED`, убедитесь, что изменения не применились.

**Шаги.**
1. Проверьте `SHOW max_prepared_transactions`. Если `0` — выведите сообщение, что демо пропущено.
2. В `twopc_demo` (id=2) вставьте значение `500`.
3. Подготовьте транзакцию: `BEGIN; INSERT ...; PREPARE TRANSACTION 'ex3_prepared'`.
4. Из другого соединения посмотрите `pg_prepared_xacts` — ваша транзакция должна быть в списке.
5. Выполните `ROLLBACK PREPARED 'ex3_prepared'`.
6. Проверьте `SELECT * FROM twopc_demo WHERE id = 2` — пусто (изменения откатились).

**Критерий успеха.** В `pg_prepared_xacts` есть `ex3_prepared` до `ROLLBACK PREPARED`; после отката `SELECT` пуст.


In [7]:
# УПРАЖНЕНИЕ 3

# TODO: проверить SHOW max_prepared_transactions
# TODO: BEGIN + INSERT (id=2, val=500) + PREPARE TRANSACTION 'ex3_prepared'
# TODO: SELECT gid FROM pg_prepared_xacts
# TODO: ROLLBACK PREPARED 'ex3_prepared'
# TODO: проверить, что строка не вставилась


## 2.4 Уровни изоляции в PostgreSQL

Аномалии чтения:

| Аномалия | Описание |
|---|---|
| **Dirty Read** | Чтение данных, изменённых ещё не зафиксированной транзакцией |
| **Non-repeatable Read** | Повторный `SELECT` той же строки возвращает другое значение |
| **Phantom Read** | Повторный запрос по диапазону возвращает другой набор строк |
| **Lost Update** | Две транзакции читают одно значение, обе пишут — одна затирает другую |

Уровни изоляции **в реализации PostgreSQL**:

1. **READ UNCOMMITTED** — автоматически повышается до `READ COMMITTED`. Никогда не используется.
2. **READ COMMITTED** (default) — свежий snapshot на каждый **оператор**.
   - Закрывает: Dirty Read.
   - **Не** закрывает: Non-repeatable Read, Phantom Read, Lost Update.
3. **REPEATABLE READ** — один snapshot на всю транзакцию (создаётся на первом запросе, не на `BEGIN`).
   - Закрывает: Dirty Read, Non-repeatable Read, **Phantom Read** (отличие от стандарта SQL: в PG phantom закрыт).
   - **Не** закрывает: Lost Update.
   - При попытке изменить строку, изменённую параллельной транзакцией после snapshot’а: `ERROR: could not serialize access due to concurrent update`.
4. **SERIALIZABLE** — SSI (Serializable Snapshot Isolation): отслеживает read/write-зависимости, откатывает при обнаружении цикла.
   - Закрывает всё, включая Lost Update и Write Skew.
   - Самая дорогая по CPU/памяти; используйте там, где действительно нужна полная сериализуемость.

**Шпаргалка:**

| Уровень | Dirty Read | Non-repeatable | Phantom | Lost Update |
|---|---|---|---|---|
| READ UNCOMMITTED (PG) | ✗ | ✓ | ✓ | ✓ |
| READ COMMITTED | ✗ | ✓ | ✓ | ✓ |
| REPEATABLE READ | ✗ | ✗ | ✗ | ✓ |
| SERIALIZABLE | ✗ | ✗ | ✗ | ✗ |

(✗ — аномалия невозможна, ✓ — возможна)


In [ ]:
# Демо: установка уровня изоляции и проверка
# Уровень по умолчанию
with engine.connect() as conn:
    iso = conn.execute(text("SHOW transaction_isolation")).scalar()
    print("Default isolation:", iso)

# Через SQLAlchemy execution_options
for level in ["READ COMMITTED", "REPEATABLE READ", "SERIALIZABLE"]:
    with engine.connect().execution_options(isolation_level=level) as conn:
        iso = conn.execute(text("SHOW transaction_isolation")).scalar()
        print(f"  requested={level!r:20s} -> actual={iso!r}")


### Упражнение 4. Non-repeatable Read на READ COMMITTED

**Задание.** Воспроизведите Non-repeatable Read на `READ COMMITTED` и убедитесь, что `REPEATABLE READ` её закрывает.

**Шаги.**
1. Создайте таблицу `ex4_accounts (id INT PRIMARY KEY, balance INT)` и вставьте `(1, 1000)`.
2. В двух потоках (`threading.Thread`):
   - **Поток A** открывает транзакцию с `READ COMMITTED`, читает `balance` (1000), спит 1 сек, читает ещё раз, печатает оба значения, делает `COMMIT`.
   - **Поток B** спит 0.3 сек, делает `UPDATE ... SET balance = 500 WHERE id = 1`, `COMMIT`.
3. Запустите и убедитесь, что второй `SELECT` в потоке A вернул **500** (non-repeatable read).
4. Повторите эксперимент, установив в потоке A `REPEATABLE READ`. Второй `SELECT` должен вернуть **1000**.
5. В комментарии: почему `REPEATABLE READ` закрывает эту аномалию?

**Критерий успеха.** В первом прогоне второе значение = 500, во втором = 1000.


In [9]:
# УПРАЖНЕНИЕ 4

import threading, time

# TODO: подготовьте ex4_accounts и строку (1, 1000)

def reader(isolation_level):
    # TODO: открыть engine.connect().execution_options(isolation_level=...)
    # TODO: SELECT balance -> print
    # TODO: time.sleep(1)
    # TODO: SELECT balance -> print
    pass

def writer():
    # TODO: time.sleep(0.3)
    # TODO: UPDATE balance = 500 WHERE id = 1; COMMIT
    pass

# TODO: запустить два потока и проверить
# TODO: повторить с REPEATABLE READ


## 2.5 Best practices по транзакциям

1. **Короткие транзакции.** Пока транзакция открыта, autovacuum не может очистить старые версии строк. Возникает bloat и растёт риск wraparound.
2. **Единый порядок блокировок.** Если обновляете несколько строк/таблиц — всегда в одном и том же порядке. Иначе почти гарантированно получите deadlock (см. раздел 5).
3. **Аналитика (OLAP) — не в OLTP-базе.**
   - Идеально — вынести на streaming-реплику.
   - Если всё же на основной: `BEGIN ISOLATION LEVEL SERIALIZABLE READ ONLY DEFERRABLE;` — PG отложит старт, чтобы не конфликтовать с параллельными SERIALIZABLE.
4. **Явные блокировки — точечно.** `FOR UPDATE` превращает optimistic concurrency в pessimistic; применяйте только там, где действительно нужно (списание денег, инвентарь, бронирование).
5. **Обрабатывайте serialization failures.** На `REPEATABLE READ` / `SERIALIZABLE` приложение **обязано** уметь повторять транзакцию при ошибке.


In [ ]:
# Демо: диагностика долгих транзакций
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT pid,
                   now() - xact_start AS duration,
                   state,
                   left(query, 60) AS query_head
            FROM pg_stat_activity
            WHERE xact_start IS NOT NULL
              AND backend_type = 'client backend'
              AND pid <> pg_backend_pid()
            ORDER BY duration DESC NULLS LAST
            LIMIT 10
        """)
        rows = cur.fetchall()
        print("Долгие транзакции (pid, duration, state, query):")
        for r in rows:
            print(" ", r)
        if not rows:
            print("  (нет других активных транзакций)")


### Упражнение 5. Диагностика долгих транзакций

**Задание.** Напишите запрос, который находит все транзакции, открытые дольше 1 секунды, и показывает их `pid`, длительность, `state`, `backend_xmin` и начало запроса.

**Шаги.**
1. Напишите SQL с фильтром `now() - xact_start > interval '1 second'`.
2. Выполните его через `psycopg` — напечатайте все строки.
3. Создайте в соседнем соединении долгую транзакцию (`BEGIN REPEATABLE READ; SELECT 1;` и подержите её через `time.sleep(2)`), чтобы проверить, что запрос её находит.
4. В комментарии: какие ещё поля `pg_stat_activity` полезны для диагностики блокировок?

**Критерий успеха.** Ваш «долгий» `SELECT` из п.3 попадает в вывод.


In [11]:
# УПРАЖНЕНИЕ 5

import time

# TODO: SQL-запрос: транзакции старше 1 сек
# TODO: выполнить через psycopg и напечатать
# TODO: в фоновом потоке открыть долгую транзакцию и проверить
# TODO: комментарий: какие ещё поля pg_stat_activity полезны


## 3. Пессимистичные блокировки и SQLAlchemy

MVCC защищает от грязного чтения, но **не спасает от Lost Update** в бизнес-логике: два пользователя прочитали одно значение, оба записали — побеждает последний.

**Пессимистичный подход** — заблокировать строку до конца транзакции через `SELECT … FOR UPDATE`.

- `SELECT … FOR UPDATE` блокирует возвращённые строки до `COMMIT`/`ROLLBACK`.
- Другие транзакции, которые тоже запросят `FOR UPDATE` (или `UPDATE`/`DELETE`) на этих строках, будут ждать.
- Обычный `SELECT` без `FOR UPDATE` **не блокируется** — читатели не страдают.
- `FOR UPDATE` дополнительно заставляет транзакцию получить XID.

**Ключевые конструкции:**
- `SET LOCAL lock_timeout = '3s'` — таймаут ожидания блокировки только до конца текущей транзакции.
- `engine.begin()` — контекстный менеджер «соединение + транзакция»: `commit()` при успехе, `rollback()` при исключении.


In [ ]:
from sqlalchemy.exc import OperationalError

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("CREATE TABLE accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO accounts VALUES (1, 1000)"))

def withdraw_money(account_id, amount):
    try:
        with engine.begin() as conn:
            conn.execute(text("SET LOCAL lock_timeout = '3s'"))
            row = conn.execute(
                text("SELECT balance FROM accounts WHERE id = :id FOR UPDATE"),
                {"id": account_id},
            ).fetchone()
            current = row[0]
            print(f"[pid-{account_id}] баланс={current}, списываем {amount}")
            if current >= amount:
                conn.execute(
                    text("UPDATE accounts SET balance = :b WHERE id = :id"),
                    {"b": current - amount, "id": account_id},
                )
                print("  списание успешно")
            else:
                print("  недостаточно средств")
    except OperationalError as e:
        print(f"[lock timeout] {type(e).__name__}: {str(e).splitlines()[0]}")

# Одиночный вызов — проходит без ожидания
withdraw_money(1, 100)

# Проверим состояние
with engine.connect() as conn:
    print("balance =", conn.execute(text("SELECT balance FROM accounts WHERE id=1")).scalar())


### Упражнение 6. Сериализация через `FOR UPDATE`

**Задание.** Убедитесь, что два параллельных `withdraw_money` выстраиваются в очередь, а не затирают друг друга.

**Шаги.**
1. Сбросьте `accounts`: `UPDATE accounts SET balance = 1000 WHERE id = 1`.
2. Запустите два потока, каждый вызывает `withdraw_money(1, 100)`. Оба должны завершиться **успешно** (баланс 1000 ≥ 200).
3. В конце проверьте: `SELECT balance FROM accounts WHERE id = 1` — должно быть **800** (1000 − 100 − 100). Это доказывает, что второй поток дождался первого и увидел уже обновлённый баланс.
4. Дополнительно: измените `withdraw_money` так, чтобы первый вызов держал транзакцию дольше (`time.sleep(4)` после `FOR UPDATE`). Второй поток должен упасть по `lock_timeout = '3s'`.
5. В комментарии: в каком бизнес-сценарии `FOR UPDATE` необходим, а в каком — избыточен?

**Критерий успеха.** При обычном запуске двух потоков баланс = 800. При искусственной задержке — второй падает по таймауту.


In [13]:
# УПРАЖНЕНИЕ 6

import threading

# TODO: сбросить balance = 1000
# TODO: запустить два потока withdraw_money(1, 100)
# TODO: дождаться, проверить баланс (должен быть 800)
# TODO: добавить sleep(4) после FOR UPDATE в первом потоке и показать lock_timeout
# TODO: комментарий про бизнес-сценарии


## 4. Оптимистичный контроль версий

Альтернатива пессимистичным блокировкам — **оптимистичная блокировка (OCC)**. Не блокируем строки, а добавляем колонку `version` (или используем `xmin`). При чтении запоминаем версию; `UPDATE` требует, чтобы версия не изменилась:

```sql
UPDATE seminar_tickets
SET status = 'booked', version = version + 1
WHERE id = :id AND version = :expected;
```

Если `rowcount = 0` — кто-то уже изменил запись. Приложение **повторяет** попытку (retry).

Плюсы:
- нет блокировок → выше пропускная способность при низкой конкуренции;
- хорошо работает в веб-приложениях с короткими транзакциями.

Минусы:
- при высокой конкуренции много retry;
- нужна дисциплина: не забыть добавить `WHERE version = ...`.

**Ключевая идея:** проверка версии в `WHERE` превращает «чтение-запись» в одну атомарную операцию. Lost Update невозможен даже без `FOR UPDATE`.


In [ ]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS seminar_tickets"))
    conn.execute(text("""
        CREATE TABLE seminar_tickets (
            id INT PRIMARY KEY,
            status TEXT,
            version INT DEFAULT 0
        )
    """))
    conn.execute(text("INSERT INTO seminar_tickets (id, status) VALUES (1, 'free')"))

def book_ticket_optimistic(ticket_id, user_name, max_retries=3):
    for attempt in range(1, max_retries + 1):
        with engine.begin() as conn:
            row = conn.execute(
                text("SELECT status, version FROM seminar_tickets WHERE id = :id"),
                {"id": ticket_id},
            ).fetchone()
            if row is None:
                print(f"{user_name}: билет не существует")
                return
            status, current_version = row
            if status != 'free':
                print(f"{user_name}: билет уже занят ({status})")
                return
            res = conn.execute(
                text("""
                    UPDATE seminar_tickets
                    SET status = :s, version = version + 1
                    WHERE id = :id AND version = :v
                """),
                {"s": f"booked_by_{user_name}", "id": ticket_id, "v": current_version},
            )
            if res.rowcount == 0:
                print(f"{user_name}: конфликт версий, попытка {attempt}/{max_retries}")
                continue
            print(f"{user_name}: успешно забронировал")
            return
    print(f"{user_name}: не удалось за {max_retries} попыток")

book_ticket_optimistic(1, "alice")


### Упражнение 7. Гонка за билетом

**Задание.** Запустите 5 потоков, пытающихся одновременно забронировать один и тот же билет. Только один должен добиться успеха, остальные — увидеть «билет уже занят» (возможно, после retry).

**Шаги.**
1. Сбросьте `seminar_tickets`: `UPDATE seminar_tickets SET status='free', version=0 WHERE id=1`.
2. Создайте 5 потоков, каждый вызывает `book_ticket_optimistic(1, f"user_{i}")`.
3. Запустите все через `threading.Thread`, дождитесь `join()`.
4. Проверьте: `SELECT status, version FROM seminar_tickets WHERE id=1` — статус = `booked_by_user_X`, `version ≥ 1`.
5. Дополнительно: посчитайте, сколько успешных бронирований и сколько конфликтов.
6. В комментарии: почему оптимистичный подход даёт лучшую пропускную способность при низкой конкуренции, но хуже — при высокой?

**Критерий успеха.** Ровно 1 успешная бронь. В консоли видны `конфликт версий` и `билет уже занят` от остальных.


In [15]:
# УПРАЖНЕНИЕ 7

import threading

# TODO: сбросить seminar_tickets (id=1, status='free', version=0)
# TODO: запустить 5 потоков с book_ticket_optimistic(1, f"user_{i}")
# TODO: дождаться, посчитать успехи/конфликты
# TODO: проверить итоговый статус билета
# TODO: комментарий про trade-off


## 5. Deadlock (взаимная блокировка)

**Deadlock** — Транзакция A держит Ресурс 1 и ждёт Ресурс 2; Транзакция B держит Ресурс 2 и ждёт Ресурс 1. Цикл.

PostgreSQL обнаруживает цикл через **`deadlock_timeout`** (по умолчанию **1 секунда**): если транзакция ждёт блокировку дольше — строится граф ожиданий, ищется цикл. При обнаружении одна из транзакций откатывается с ошибкой `deadlock detected`.

Обратите внимание: `deadlock_timeout` (1 с) обычно **меньше** `lock_timeout` (например, 3 с), поэтому при deadlock вы увидите именно `DeadlockDetected`, а не `LockNotAvailable`.

**Как лечить:**
- Единый порядок захвата блокировок во всех транзакциях.
- Короткие транзакции.
- Обработка `OperationalError` с retry.

**Как диагностировать:**
- `pg_stat_activity` + `pg_blocking_pids(pid)` — кто кого держит.
- `pg_locks` — все блокировки.
- `SELECT pg_terminate_backend(<blocking_pid>)` — точечно завершить «виновника».


In [ ]:
import threading, time
from sqlalchemy.exc import OperationalError

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS accounts"))
    conn.execute(text("CREATE TABLE accounts (id INT PRIMARY KEY, balance INT)"))
    conn.execute(text("INSERT INTO accounts VALUES (1, 1000), (2, 2000)"))

def transaction_1():
    try:
        with engine.begin() as conn:
            print("T1: блокируем счёт 1")
            conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))
            time.sleep(1)
            print("T1: пытаемся заблокировать счёт 2...")
            conn.execute(text("SET LOCAL lock_timeout = '2s'"))
            conn.execute(text("SELECT balance FROM accounts WHERE id = 2 FOR UPDATE"))
            print("T1: успех")
    except OperationalError as e:
        print(f"T1 повалена: {type(e).__name__}: {str(e).splitlines()[0]}")

def transaction_2():
    try:
        with engine.begin() as conn:
            print("T2: блокируем счёт 2")
            conn.execute(text("SELECT balance FROM accounts WHERE id = 2 FOR UPDATE"))
            time.sleep(1)
            print("T2: пытаемся заблокировать счёт 1...")
            conn.execute(text("SET LOCAL lock_timeout = '2s'"))
            conn.execute(text("SELECT balance FROM accounts WHERE id = 1 FOR UPDATE"))
            print("T2: успех")
    except OperationalError as e:
        print(f"T2 повалена: {type(e).__name__}: {str(e).splitlines()[0]}")

t1 = threading.Thread(target=transaction_1)
t2 = threading.Thread(target=transaction_2)
t1.start(); t2.start()
t1.join(); t2.join()


### Упражнение 8. Устраняем deadlock

**Задание.** «Вылечите» демонстрацию: измените `transaction_2` так, чтобы она блокировала счета **в том же порядке**, что и `transaction_1` (сначала счёт 1, потом счёт 2). Deadlock должен исчезнуть.

**Шаги.**
1. Скопируйте демо-код в новую ячейку.
2. Измените `transaction_2`: вместо блокировки счёта 2 первым шагом блокируйте счёт 1.
3. Запустите скрипт **несколько раз** (3–5). Deadlock не должен появляться.
4. В комментарии объясните, почему единый порядок захвата ресурсов делает deadlock невозможным.
5. Дополнительно: напишите запрос `SELECT blocked.pid, blocking.pid, blocked.query FROM ...` с использованием `pg_blocking_pids(blocked.pid)` — выведите, кто кого блокирует, если один из потоков висит.

**Критерий успеха.** Обе транзакции всегда печатают `успех`. Ни в одном прогоне нет `DeadlockDetected`.


In [ ]:
# УПРАЖНЕНИЕ 8

import threading, time
from sqlalchemy.exc import OperationalError

# TODO: transaction_1 без изменений (блокирует 1, потом 2)
# TODO: transaction_2 ИСПРАВЛЕНА — блокирует 1, потом 2
# TODO: запустить 3-5 раз, убедиться что deadlock исчез
# TODO: комментарий про единый порядок блокировок
# TODO: запрос с pg_blocking_pids для диагностики
